# Module 4 | Class 1 Assignment: Regression Modeling — Sales Forecasting
**Objective:** Build a linear regression model to forecast sales, interpret its outputs, and understand when simple models are enough.

**Dataset:** Superstore Dataset (Sample - Superstore.csv)


## Task 1: Explore the Data

In [ ]:
# Import all required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

print("All libraries imported successfully!")


In [ ]:
# Step 1: Load the dataset
# Download from: https://www.kaggle.com/datasets/vivek468/superstore-dataset-final
df = pd.read_csv('Sample - Superstore.csv', encoding='latin1')
print("Dataset loaded successfully!")
print(f"Shape: {df.shape}")


In [ ]:
# Step 2: Inspect the dataset
print("=== Data Types ===")
print(df.dtypes)
print()
print("=== Statistical Summary ===")
df.describe()


In [ ]:
# Step 3: Select numerical features as predictors
features = ['Quantity', 'Discount', 'Profit']
target = 'Sales'

print(f"Selected features: {features}")
print(f"Target variable: {target}")


In [ ]:
# Step 4: Check for missing values
missing = df[features + [target]].isnull().sum()
print("=== Missing Values ===")
print(missing)

# Handle any NaNs (if present)
df_clean = df[features + [target]].dropna()
print(f"\nRows before dropping NaNs: {len(df)}")
print(f"Rows after dropping NaNs:  {len(df_clean)}")


In [ ]:
# Step 5: Correlation matrix heatmap
plt.figure(figsize=(8, 6))
corr = df_clean[['Sales', 'Quantity', 'Discount', 'Profit']].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Feature Correlations with Sales', fontsize=14)
plt.tight_layout()
plt.show()

print("\nCorrelation Matrix:")
print(corr)


## Task 2: Build a Linear Regression Model

In [ ]:
# Step 1: Define X (features) and y (target)
X = df_clean[features]
y = df_clean[target]

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")


In [ ]:
# Step 2: Split data into training and test sets (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set size: {X_train.shape[0]} rows")
print(f"Test set size:     {X_test.shape[0]} rows")


In [ ]:
# Step 3: Fit Linear Regression model
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("Linear Regression model trained successfully!")


In [ ]:
# Step 4: Compute evaluation metrics
mse  = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae  = mean_absolute_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)

print("=== Linear Regression Metrics ===")
print(f"MSE  : {mse:.2f}")
print(f"RMSE : {rmse:.2f}")
print(f"MAE  : {mae:.2f}")
print(f"R²   : {r2:.4f}")

# Save for comparison table later
linear_metrics = {'MSE': mse, 'RMSE': rmse, 'MAE': mae, 'R2': r2}


## Task 3: Interpret the Coefficients

In [ ]:
# Step 1: Print model coefficients and intercept
print("=== Model Coefficients ===")
for feature, coef in zip(X.columns, model.coef_):
    print(f"  {feature:12s}: {coef:.4f}")
print(f"  {'Intercept':12s}: {model.intercept_:.4f}")


### Coefficient Interpretation

**Profit coefficient:**  
A one-unit increase in Profit is associated with a positive increase in Sales, holding Quantity and Discount constant. This makes intuitive business sense — higher-profit items tend to have higher selling prices, directly driving up the Sales figure.

**Quantity coefficient:**  
A one-unit increase in Quantity (one more item sold) is associated with an increase in Sales proportional to the average unit price in the dataset. More items sold naturally means more revenue generated.

**Discount coefficient (potential unexpected sign):**  
The Discount coefficient may appear negative or surprisingly small. This is counterintuitive at first — one might expect higher discounts to drive higher volume and thus higher sales. However, discount is expressed as a fraction (0–1), so a higher discount directly reduces the per-unit revenue. Even if discounts increase quantity, the revenue impact can be negative overall. This reflects a classic business trade-off: discounting can hurt top-line sales even while increasing units sold.


## Task 4: Try Polynomial Regression

In [ ]:
# Step 1: Create polynomial features (degree=2)
poly = PolynomialFeatures(degree=2, include_bias=False)
X_train_poly = poly.fit_transform(X_train)
X_test_poly  = poly.transform(X_test)

print(f"Original feature count : {X_train.shape[1]}")
print(f"Polynomial feature count: {X_train_poly.shape[1]}")
print(f"\nNew feature names:")
print(poly.get_feature_names_out(features))


In [ ]:
# Step 2: Train Polynomial Regression model
model_poly = LinearRegression()
model_poly.fit(X_train_poly, y_train)
y_pred_poly = model_poly.predict(X_test_poly)

print("Polynomial Regression model trained successfully!")


In [ ]:
# Step 3: Compute metrics for Polynomial Regression
mse_p  = mean_squared_error(y_test, y_pred_poly)
rmse_p = np.sqrt(mse_p)
mae_p  = mean_absolute_error(y_test, y_pred_poly)
r2_p   = r2_score(y_test, y_pred_poly)

print("=== Polynomial Regression Metrics ===")
print(f"MSE  : {mse_p:.2f}")
print(f"RMSE : {rmse_p:.2f}")
print(f"MAE  : {mae_p:.2f}")
print(f"R²   : {r2_p:.4f}")

poly_metrics = {'MSE': mse_p, 'RMSE': rmse_p, 'MAE': mae_p, 'R2': r2_p}


In [ ]:
# Comparison table: Linear vs Polynomial
comparison = pd.DataFrame({
    'Metric': ['MSE', 'RMSE', 'MAE', 'R²'],
    'Linear Regression': [
        linear_metrics['MSE'],
        linear_metrics['RMSE'],
        linear_metrics['MAE'],
        linear_metrics['R2']
    ],
    'Polynomial Regression (deg=2)': [
        poly_metrics['MSE'],
        poly_metrics['RMSE'],
        poly_metrics['MAE'],
        poly_metrics['R2']
    ]
}).set_index('Metric')

comparison = comparison.round(4)
print("=== Model Comparison Table ===")
print(comparison.to_string())
comparison


### Did Polynomial Features Help?

By adding degree-2 polynomial features (squared terms and interaction terms such as Quantity×Profit, Discount²), the model gains the ability to capture non-linear relationships between the predictors and Sales. 

If the polynomial R² is higher than the linear R², it indicates that some non-linear structure exists in the data that the simple linear model missed. However, polynomial regression with small datasets or noisy targets can also **overfit** — showing lower error on training data but not improving (or even worsening) on the test set. 

A small or negative improvement in test R² for the polynomial model suggests that the relationship between these three features and Sales is largely linear, and the extra complexity adds noise rather than signal. In that case, the simpler linear model is preferable (Occam's Razor in ML).


## Task 5: Visualize

In [ ]:
# Step 1: Predicted vs Actual scatter plot
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.5, s=20, color='steelblue', label='Predictions')
plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    'r--', lw=2, label='Perfect prediction'
)
plt.xlabel('Actual Sales', fontsize=12)
plt.ylabel('Predicted Sales', fontsize=12)
plt.title('Predicted vs Actual Sales (Linear Regression)', fontsize=14)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Step 2: Residual plot
residuals = y_test - y_pred

plt.figure(figsize=(8, 6))
plt.scatter(y_pred, residuals, alpha=0.5, s=20, color='darkorange')
plt.axhline(y=0, color='r', linestyle='--', lw=2)
plt.xlabel('Predicted Sales', fontsize=12)
plt.ylabel('Residuals (Actual − Predicted)', fontsize=12)
plt.title('Residual Plot (Linear Regression)', fontsize=14)
plt.tight_layout()
plt.show()


### Plot Interpretation

**Predicted vs Actual plot:**  
Points lying close to the red dashed diagonal line indicate accurate predictions. A tight cluster along the diagonal means the model performs well. Scattered points far from the line — especially at higher Sales values — reveal that the model struggles to predict large sales figures, which is common when high-value outliers exist in the data.

**Residual plot:**  
Random scatter of residuals around the horizontal zero line is the ideal outcome, indicating no systematic bias. If a cone shape (fan-out) is visible — where residuals grow larger as predicted values increase — this is **heteroscedasticity**, meaning the variance of errors is not constant. For this sales dataset, such a pattern is expected because high-value transactions introduce more variability. A U-shaped or curved pattern would indicate a non-linear relationship that the linear model is missing entirely.


## Summary

In [ ]:
# Final summary of both models
print("=" * 55)
print("         FINAL MODEL COMPARISON SUMMARY")
print("=" * 55)
print(f"{'Metric':<8} {'Linear Reg':>15} {'Poly Reg (deg=2)':>18}")
print("-" * 55)
for metric in ['MSE', 'RMSE', 'MAE', 'R2']:
    lv = linear_metrics[metric]
    pv = poly_metrics[metric]
    print(f"{metric:<8} {lv:>15.4f} {pv:>18.4f}")
print("=" * 55)
print()
better = "Polynomial" if poly_metrics['R2'] > linear_metrics['R2'] else "Linear"
print(f"Better performing model (by R²): {better} Regression")
